<a href="https://colab.research.google.com/github/ANUDHEERPARIMI/PyTorch/blob/PyTorch/fasion_minist_dataset_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
import torch

In [2]:
torch.manual_seed(42)

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fashionmnist


In [5]:
import os
path = "/kaggle/input/fashionmnist"
print(os.listdir(path))


['t10k-labels-idx1-ubyte', 't10k-images-idx3-ubyte', 'fashion-mnist_test.csv', 'fashion-mnist_train.csv', 'train-labels-idx1-ubyte', 'train-images-idx3-ubyte']


In [8]:
df=pd.read_csv("/kaggle/input/fashionmnist/fashion-mnist_train.csv")

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
df.shape

(60000, 785)

In [11]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [12]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [13]:
y

array([2, 9, 6, ..., 8, 8, 7])

In [14]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [15]:
X_train=X_train/255.0
X_test=X_test/255.0

In [16]:
class CustomDataset(Dataset):
  def __init__(self,features,label):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.label=torch.tensor(label,dtype=torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,idx):
    return self.features[idx],self.label[idx]

In [21]:
train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)

In [22]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)


In [23]:
class MyNN(nn.Module):
  def __init__(self,features):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(features, 128),
      nn.ReLU(),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Linear(64, 10)
  )

  def forward(self, x):
      return self.layers(x)



In [24]:
epochs=100
learning_rate=0.1

In [25]:
model=MyNN(X_train.shape[1])
model=model.to(device)

criterian = nn.CrossEntropyLoss()

optmizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

In [28]:
for epoch in range(epochs):
  total_loss=0
  for batch_features,batch_labels in train_loader:

    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    y_pred=model(batch_features)

    loss = criterian(y_pred,batch_labels)

    optmizer.zero_grad()

    loss.backward()

    optmizer.step()

    total_loss+=loss.item()
  print(f"{epoch+1} for this this {total_loss/len(train_loader)}")

1 for this this 0.3024805247411132
2 for this this 0.28907031117379667
3 for this this 0.2813529353141785
4 for this this 0.2724223312412699
5 for this this 0.262374625792106
6 for this this 0.2539863397950927
7 for this this 0.24757420970934133
8 for this this 0.23973180836687485
9 for this this 0.23530911072716118
10 for this this 0.22863688788563014
11 for this this 0.22005892945018907
12 for this this 0.2187483137883246
13 for this this 0.21311368865519761
14 for this this 0.20871275233229
15 for this this 0.20215174007043243
16 for this this 0.19998510887846352
17 for this this 0.1919950239441047
18 for this this 0.18902553340420128
19 for this this 0.1850539008155465
20 for this this 0.1793009690164278
21 for this this 0.17859762749572594
22 for this this 0.17592150477630397
23 for this this 0.17191476952905457
24 for this this 0.1706115890974179
25 for this this 0.16408878175821157
26 for this this 0.1614479704533393
27 for this this 0.1579093978268405
28 for this this 0.1554267

In [29]:
model.eval()

MyNN(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [30]:
total=0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in test_loader:

    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _,predicted=torch.max(outputs,1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted==batch_labels).sum().item()


In [31]:
print(correct/total)

0.8933333333333333


In [32]:
total

12000

In [33]:
y_test.shape

(12000,)

In [34]:
for x,y in test_loader:
  print(y)
  break

tensor([7, 8, 8, 5, 9, 1, 2, 6, 6, 2, 5, 0, 7, 1, 6, 0, 6, 2, 9, 1, 2, 4, 8, 0,
        4, 9, 1, 0, 0, 5, 1, 6])
